In [8]:
import numpy as np
import pandas as pd
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

LAG_PREFIXES = ["lag5", "lag10", "lag21", "lag63"]
TARGET_PREFIX = "target"

INPUT_FILES = {
    10: "../proxy/realized_cov_target_h10_lag_5_10_21_63.csv",
    21: "../proxy/realized_cov_target_h21_lag_5_10_21_63.csv",
    63: "../proxy/realized_cov_target_h63_lag_5_10_21_63.csv",
}

OUTPUT_TEMPLATE = "../proxy/realized_chol_target_h{h}_lag_5_10_21_63.csv"

Grid search for SVR 

In [14]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =========================================================
# GRID SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average Frobenius distance
# Validation period: 2019 only
# Compare against realized covariance proxy
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 21

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2019-12-31"

WINDOW_GRID = [100, 150, 200, 250, 300, 320]
C_GRID = [0.1, 1, 10]
GAMMA_GRID = ["scale", 0.1]
EPSILON_GRID = [0.001]

LAGS = (10, 21, 63)
REFIT_EVERY = 1

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = pd.read_csv(CHOL_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
proxy = pd.read_csv(PROXY_PATH, parse_dates=["Date"]).sort_values("Date").set_index("Date")

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):

    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:

        p_loc = chol.index[chol["Date"] == p][0]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:

            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:

                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )

                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]

        L = np.zeros((n_assets, n_assets))

        for a, b in lower_pairs:

            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs = scaler_x.transform(X)
            y_scaled = model.predict(Xs)

            y_pred = scaler_y.inverse_transform(
                y_scaled.reshape(-1, 1)
            )[0, 0]

            i = ASSET_ORDER.index(a)
            j = ASSET_ORDER.index(b)

            L[i, j] = y_pred

        Sigma = L @ L.T

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)

        frob = frobenius_distance(S_true, Sigma)

        if np.isfinite(frob):
            frob_list.append(frob)
            n_forecasts += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_frobenius": np.mean(frob_list),
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# BUILD GRID
# ---------------------------------------------------------
specs = list(product(WINDOW_GRID, C_GRID, GAMMA_GRID, EPSILON_GRID))

print("Total SVR specs:", len(specs))
print()

# ---------------------------------------------------------
# GRID SEARCH
# ---------------------------------------------------------
results = []

for window, C, gamma, epsilon in tqdm(specs, desc=f"SVR grid search (h={TARGET_H})"):

    res = evaluate_svr_spec(window, C, gamma, epsilon)

    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# RANK MODELS
# ---------------------------------------------------------
results_df = results_df.sort_values(
    ["avg_frobenius", "window"],
    ascending=[True, True]
).reset_index(drop=True)

print("\n=========================================================")
print("SVR MODELS RANKED BY FROBENIUS DISTANCE")
print("Forecast horizon:", TARGET_H)
print("Validation period: 2019")
print("=========================================================\n")

print(results_df.to_string(index=False))

best = results_df.iloc[0]

print("\n=========================================================")
print("BEST SVR MODEL")
print("=========================================================")
print(best.to_string())

Loaded Cholesky data: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h21.csv
Validation: 2019-01-02 00:00:00 -> 2019-12-31 00:00:00

Total SVR specs: 36



SVR grid search (h=21):   0%|          | 0/36 [00:00<?, ?it/s]


SVR MODELS RANKED BY FROBENIUS DISTANCE
Forecast horizon: 21
Validation period: 2019

 window    C gamma  epsilon  avg_frobenius  n_forecasts  n_refits
    320  1.0   0.1    0.001       0.001160          252       252
    200  0.1   0.1    0.001       0.001162          252       252
    200  0.1 scale    0.001       0.001165          252       252
    200  1.0   0.1    0.001       0.001169          252       252
    150  0.1   0.1    0.001       0.001170          252       252
    320  0.1 scale    0.001       0.001176          252       252
    250  1.0 scale    0.001       0.001189          252       252
    300  0.1 scale    0.001       0.001192          252       252
    250  0.1 scale    0.001       0.001197          252       252
    320  0.1   0.1    0.001       0.001198          252       252
    300  0.1   0.1    0.001       0.001198          252       252
    320  1.0 scale    0.001       0.001201          252       252
    150  0.1 scale    0.001       0.001205          252

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

LAG_PREFIXES = ["lag5", "lag10", "lag21", "lag63"]
TARGET_PREFIX = "target"

INPUT_FILES = {
    10: "../proxy/realized_cov_target_h10_lag_5_10_21_63.csv",
    21: "../proxy/realized_cov_target_h21_lag_5_10_21_63.csv",
    63: "../proxy/realized_cov_target_h63_lag_5_10_21_63.csv",
}

OUTPUT_TEMPLATE = "../proxy/realized_chol_target_h{h}_lag_5_10_21_63.csv"

# Small ridge for numerical stability before Cholesky
BASE_RIDGE = 1e-12
MAX_RIDGE_TRIES = 8


# =========================================================
# HELPERS
# =========================================================
def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_cov_cols(prefix, asset_order):
    return [f"{prefix}_cov_{a}__{b}" for a, b in build_full_pairs(asset_order)]


def build_chol_cols(prefix, asset_order):
    return [f"{prefix}_chol_{a}__{b}" for a, b in build_lower_tri_pairs(asset_order)]


def extract_cov_matrix_from_row(row, prefix, asset_order):
    n = len(asset_order)
    mat = np.zeros((n, n), dtype=float)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            mat[i, j] = row[col]

    # Force exact symmetry in case of tiny numerical mismatch
    mat = 0.5 * (mat + mat.T)
    return mat


def safe_cholesky(mat, base_ridge=1e-12, max_tries=8):
    """
    Try Cholesky with increasing ridge if needed.
    Returns lower-triangular matrix L.
    """
    n = mat.shape[0]
    eye = np.eye(n)
    ridge = base_ridge

    for _ in range(max_tries):
        try:
            return np.linalg.cholesky(mat + ridge * eye)
        except np.linalg.LinAlgError:
            ridge *= 10

    min_eig = np.min(np.linalg.eigvalsh(mat))
    raise np.linalg.LinAlgError(
        f"Cholesky failed even after ridge escalation. "
        f"Minimum eigenvalue before ridge: {min_eig:.6e}"
    )


def flatten_cholesky_lower_tri(L, asset_order):
    out = {}

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                out[f"{a}__{b}"] = float(L[i, j])

    return out


def get_metadata_columns(df):
    preferred = [
        "Date",
        "lag5_start", "lag5_end",
        "lag10_start", "lag10_end",
        "lag21_start", "lag21_end",
        "lag63_start", "lag63_end",
        "target_start", "target_end",
    ]
    return [c for c in preferred if c in df.columns]


def convert_cov_df_to_chol_df(df_cov, asset_order,
                              base_ridge=1e-12, max_ridge_tries=8):
    metadata_cols = get_metadata_columns(df_cov)
    prefixes = [TARGET_PREFIX] + LAG_PREFIXES

    expected_cov_cols = []
    for prefix in prefixes:
        expected_cov_cols.extend(build_cov_cols(prefix, asset_order))

    missing = [c for c in expected_cov_cols if c not in df_cov.columns]
    if missing:
        raise ValueError(f"Missing expected covariance columns, e.g. {missing[:10]}")

    output_rows = []

    for row_idx, (_, row) in enumerate(df_cov.iterrows()):
        out_row = {}

        # Keep metadata
        for c in metadata_cols:
            out_row[c] = row[c]

        # Convert each covariance block to lower-triangular Cholesky entries
        for prefix in prefixes:
            cov_mat = extract_cov_matrix_from_row(row, prefix, asset_order)
            L = safe_cholesky(
                cov_mat,
                base_ridge=base_ridge,
                max_tries=max_ridge_tries
            )

            chol_entries = flatten_cholesky_lower_tri(L, asset_order)

            for pair_name, value in chol_entries.items():
                out_row[f"{prefix}_chol_{pair_name}"] = value

        output_rows.append(out_row)

        if (row_idx + 1) % 250 == 0:
            print(f"Processed {row_idx + 1} rows...")

    df_chol = pd.DataFrame(output_rows)

    # Final column order
    final_cols = metadata_cols.copy()
    for prefix in prefixes:
        final_cols.extend(build_chol_cols(prefix, asset_order))

    df_chol = df_chol[fina l_cols].copy()
    return df_chol


def convert_one_horizon(h, input_path, output_path, asset_order,
                        base_ridge=1e-12, max_ridge_tries=8):
    print("=" * 70)
    print(f"Converting horizon h={h}")
    print("Input :", input_path)
    print("Output:", output_path)

    df_cov = pd.read_csv(input_path, parse_dates=["Date"])
    print("Loaded shape:", df_cov.shape)

    df_chol = convert_cov_df_to_chol_df(
        df_cov=df_cov,
        asset_order=asset_order,
        base_ridge=base_ridge,
        max_ridge_tries=max_ridge_tries
    )

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_chol.to_csv(output_path, index=False)

    print("Saved shape:", df_chol.shape)
    print("Saved to   :", output_path)
    print()


# =========================================================
# RUN FOR ALL HORIZONS
# =========================================================
for h, input_path in INPUT_FILES.items():
    output_path = OUTPUT_TEMPLATE.format(h=h)

    convert_one_horizon(
        h=h,
        input_path=input_path,
        output_path=output_path,
        asset_order=ASSET_ORDER,
        base_ridge=BASE_RIDGE,
        max_ridge_tries=MAX_RIDGE_TRIES
    )

Converting horizon h=10
Input : ../proxy/realized_cov_target_h10_lag_5_10_21_63.csv
Output: ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv
Loaded shape: (2132, 331)
Processed 250 rows...
Processed 500 rows...
Processed 750 rows...
Processed 1000 rows...
Processed 1250 rows...
Processed 1500 rows...
Processed 1750 rows...
Processed 2000 rows...
Saved shape: (2132, 191)
Saved to   : ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv

Converting horizon h=21
Input : ../proxy/realized_cov_target_h21_lag_5_10_21_63.csv
Output: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded shape: (2121, 331)
Processed 250 rows...
Processed 500 rows...
Processed 750 rows...
Processed 1000 rows...
Processed 1250 rows...
Processed 1500 rows...
Processed 1750 rows...
Processed 2000 rows...
Saved shape: (2121, 191)
Saved to   : ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv

Converting horizon h=63
Input : ../proxy/realized_cov_target_h63_lag_5_10_21_63.csv
Output: ../proxy/realized_ch

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 21

ASSET_ORDER = [
    "US_EQUITY","INTL_EQUITY","US_BONDS","INTL_BONDS",
    "US_REIT","INTL_REIT","GOLD","BTC"
]

LAGS = (10,21,63)

TEST_START = "2021-01-01"

TRAINING_WINDOW = 320
REFIT_EVERY = 1

KERNEL = "rbf"
C = 1.0
EPSILON = 0.001
GAMMA = 0.1


CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
SAVE_PATH = f"../results/svr_cov_forecast_h{TARGET_H}.csv"


# =========================================================
# HELPERS
# =========================================================
def build_lower_pairs(asset_order):

    pairs = []
    for i,a in enumerate(asset_order):
        for j,b in enumerate(asset_order):
            if i >= j:
                pairs.append((a,b))
    return pairs


def build_full_pairs(asset_order):

    return [(a,b) for a in asset_order for b in asset_order]


def reconstruct_covariance(L):

    return L @ L.T


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CHOL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

lower_pairs = build_lower_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

test_start = pd.Timestamp(TEST_START)

candidate_dates = df.loc[df["Date"] >= test_start,"Date"]


# =========================================================
# ROLLING FORECAST
# =========================================================
forecast_rows = []

models = None
scalers_x = None
scalers_y = None
last_refit_index = None


for p in candidate_dates:

    p_idx = df.index[df["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_index is None
        or (p_idx - last_refit_index) >= REFIT_EVERY
    )

    if should_refit:

        train_end = p_idx - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = df.iloc[train_start:train_end+1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a,b in lower_pairs:

            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1,1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = SVR(
                kernel=KERNEL,
                C=C,
                epsilon=EPSILON,
                gamma=GAMMA
            )

            model.fit(Xs,ys)

            models[(a,b)] = model
            scalers_x[(a,b)] = scaler_x
            scalers_y[(a,b)] = scaler_y

        last_refit_index = p_idx

        print(
            f"Refit at {p.date()} | "
            f"train rows {train.index.min()} -> {train.index.max()}"
        )


    # =====================================================
    # PREDICT CHOLESKY FACTOR
    # =====================================================
    n = len(ASSET_ORDER)
    L_pred = np.zeros((n,n))

    row = df.iloc[p_idx]

    for a,b in lower_pairs:

        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_pred = row[lag_cols].values.reshape(1,-1)

        scaler_x = scalers_x[(a,b)]
        scaler_y = scalers_y[(a,b)]
        model = models[(a,b)]

        Xs = scaler_x.transform(X_pred)

        y_scaled = model.predict(Xs)

        y_pred = scaler_y.inverse_transform(
            y_scaled.reshape(-1,1)
        )[0,0]

        i = ASSET_ORDER.index(a)
        j = ASSET_ORDER.index(b)

        L_pred[i,j] = y_pred


    # =====================================================
    # RECONSTRUCT COVARIANCE
    # =====================================================
    Sigma = reconstruct_covariance(L_pred)

    out_row = {"Date":p}

    for i,a in enumerate(ASSET_ORDER):
        for j,b in enumerate(ASSET_ORDER):

            out_row[f"cov_{a}__{b}"] = Sigma[i,j]

    forecast_rows.append(out_row)


# =========================================================
# SAVE FORECASTS
# =========================================================
forecast_df = pd.DataFrame(forecast_rows)

cols = ["Date"] + [
    f"cov_{a}__{b}" for a,b in full_pairs
]

forecast_df = forecast_df[cols]

Path("../results").mkdir(exist_ok=True)

forecast_df.to_csv(SAVE_PATH,index=False)

print()
print("Saved forecasts to:",SAVE_PATH)
print("Shape:",forecast_df.shape)

Refit at 2021-01-04 | train rows 546 -> 865
Refit at 2021-01-05 | train rows 547 -> 866
Refit at 2021-01-06 | train rows 548 -> 867
Refit at 2021-01-07 | train rows 549 -> 868
Refit at 2021-01-08 | train rows 550 -> 869
Refit at 2021-01-11 | train rows 551 -> 870
Refit at 2021-01-12 | train rows 552 -> 871
Refit at 2021-01-13 | train rows 553 -> 872
Refit at 2021-01-14 | train rows 554 -> 873
Refit at 2021-01-15 | train rows 555 -> 874
Refit at 2021-01-19 | train rows 556 -> 875
Refit at 2021-01-20 | train rows 557 -> 876
Refit at 2021-01-21 | train rows 558 -> 877
Refit at 2021-01-22 | train rows 559 -> 878
Refit at 2021-01-25 | train rows 560 -> 879
Refit at 2021-01-26 | train rows 561 -> 880
Refit at 2021-01-27 | train rows 562 -> 881
Refit at 2021-01-28 | train rows 563 -> 882
Refit at 2021-01-29 | train rows 564 -> 883
Refit at 2021-02-01 | train rows 565 -> 884
Refit at 2021-02-02 | train rows 566 -> 885
Refit at 2021-02-03 | train rows 567 -> 886
Refit at 2021-02-04 | train rows